### Tutorial 09: Create Lens

In [11]:
import bpy
import math
import numpy as np
from IPython.display import display, Image
import matplotlib.pyplot as plt
import import_ipynb

from L001_hello_world import render, clear_scene

# Clear scene
if __name__ == '__main__':
    clear_scene()

    bpy.ops.preferences.addon_enable(module="bl_ext.blender_org.extra_mesh_objects")

In [12]:
# Set render engine for ray tracing
bpy.context.scene.render.engine = 'CYCLES'
bpy.context.scene.render.resolution_x = 512
bpy.context.scene.render.resolution_y = 512
bpy.context.scene.view_settings.view_transform = 'Raw'
bpy.context.scene.cycles.use_adaptive_sampling = False
bpy.context.scene.cycles.samples = 100
bpy.context.scene.cycles.use_denoising = False

In [13]:
# Create camera
bpy.ops.object.camera_add(rotation=(np.pi/2, 0, 0),location=(0,-3.3,-0.7))
cam = bpy.context.active_object
bpy.context.scene.camera = cam

# Add plane to scene
bpy.ops.mesh.primitive_plane_add(location=(0, 1.25, -0.7), rotation=(np.pi/2, 0, 0))
plane = bpy.context.active_object

In [14]:
file_path = "MAP10100100-A-Zemax.zmx"

class Lens:
    def __init__(self):
        self.curv = 0
        self.radi = 1
        self.disz = 0
        self.glas = 1

materials = {}
materials['N-BK7'] = 1.51680
materials['SF5'] = 1.67270

lenses = []

with open(file_path, 'r') as file:
    for line in file:
        parts = line.split()
        command = parts[0]
        arguments = parts[1:]

        if (command == 'SURF'):
            lenses.append(Lens())
        elif (command == 'CURV'):
            lenses[-1].curv = float(arguments[0])*10
        elif (command == 'DIAM'):
            lenses[-1].radi = 0.5*float(arguments[0])/10
        elif (command == 'DISZ'):
            lenses[-1].disz = float(arguments[0])/10
        elif (command == 'GLAS'):
            lenses[-1].glas = materials[arguments[0]]

dist = 0
prev_refr = 1
curr_refr = 1

prev_pos = 1
curr_pos = 1

for itr in range(len(lenses)):
    if (math.isinf(lenses[itr].disz)):
        continue

    curv = lenses[itr].curv
    curr_refr = lenses[itr].glas/prev_refr
    prev_refr = lenses[itr].glas

    if (np.abs(curv) <= 1e-6):
        continue

    x_eq = "v*cos(u)"
    y_eq = "v*sin(u)"
    z_eq = str(1/curv) + "*(sqrt(1-(v*" + str(curv) + ")**2)-1)-" + str(dist)

    bpy.ops.mesh.primitive_xyz_function_surface(x_eq=x_eq, y_eq=y_eq, z_eq=z_eq, range_u_min=0, range_u_max=2*np.pi, range_u_step=32, wrap_u=True, range_v_min=0, range_v_max=lenses[itr].radi, range_v_step=16, wrap_v=False, close_v=False)

    prev_pos = curr_pos
    curr_pos = (1/curv) * (np.sqrt(1 - (lenses[itr].radi*curv)**2)-1) - dist

    bpy.ops.object.mode_set(mode='OBJECT')

    # Get the created surface object
    surface = bpy.context.active_object
    bpy.ops.object.shade_smooth()

    # Create a new material
    material = bpy.data.materials.new(name="Refractive Material")
    material.use_nodes = True
    nodes = material.node_tree.nodes
    nodes.clear()

    # Set up a simple Principled BSDF shader
    refraction_node = nodes.new(type='ShaderNodeBsdfRefraction')
    refraction_node.inputs['IOR'].default_value = 1/curr_refr
    output_node = nodes.new(type='ShaderNodeOutputMaterial')

    links = material.node_tree.links
    links.new(refraction_node.outputs['BSDF'], output_node.inputs['Surface'])

    # Assign the material to the surface
    if surface.data.materials:
        surface.data.materials[0] = material
    else:
        surface.data.materials.append(material)


    if (np.abs(prev_refr - curr_refr) > 1e-3):
        x_eq = "cos(u)*(" + str(lenses[itr-1].radi) + "*v + " + str(lenses[itr].radi) + "*(1-v))"
        y_eq = "sin(u)*(" + str(lenses[itr-1].radi) + "*v + " + str(lenses[itr].radi) + "*(1-v))"
        z_eq = str(prev_pos) + "*v + " + str(curr_pos) + "*(1-v)"

        bpy.ops.mesh.primitive_xyz_function_surface(x_eq=x_eq, y_eq=y_eq, z_eq=z_eq, range_u_min=0, range_u_max=2*np.pi, range_u_step=32, wrap_u=True, range_v_min=0, range_v_max=1, range_v_step=1, wrap_v=False, close_v=False)

        bpy.ops.object.mode_set(mode='OBJECT')

        # Get the created surface object
        surface = bpy.context.active_object
        bpy.ops.object.shade_smooth()


        # Create a new material
        material = bpy.data.materials.new(name="Refractive Material")
        material.use_nodes = True
        nodes = material.node_tree.nodes
        nodes.clear()

        # Set up a simple Principled BSDF shader
        refraction_node = nodes.new(type='ShaderNodeBsdfRefraction')
        refraction_node.inputs['IOR'].default_value = lenses[itr-1].glas
        output_node = nodes.new(type='ShaderNodeOutputMaterial')

        links = material.node_tree.links
        links.new(refraction_node.outputs['BSDF'], output_node.inputs['Surface'])

        # Assign the material to the surface
        if surface.data.materials:
            surface.data.materials[0] = material
        else:
            surface.data.materials.append(material)

    dist = dist + lenses[itr].disz



/var/folders/fg/nxhkxc3d22s15cc0vxm03vxh0000gn/T/ipykernel_15929/513578676.py:68: DeprecationWarning: 'Material.use_nodes' is expected to be removed in Blender 6.0
  material.use_nodes = True
/var/folders/fg/nxhkxc3d22s15cc0vxm03vxh0000gn/T/ipykernel_15929/513578676.py:103: DeprecationWarning: 'Material.use_nodes' is expected to be removed in Blender 6.0
  material.use_nodes = True
